In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
# Change this to the folder that contains all participant folders
ROOT_DIR = Path("/Users/joanna.luberadzka/Projects/2026/chataid-web-pilot-analysis/data")


rows = []

for participant_dir in ROOT_DIR.iterdir():

    if participant_dir.name == "experiment_Joanna_1779782845088":
        continue

    if not participant_dir.is_dir():
        continue

    json_path = participant_dir / "experiment_data_plus_analysis.json"

    if not json_path.exists():
        print(f"Skipping {participant_dir.name}: no experiment_data_plus_analysis.json found")
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    participant_info = data.get("participant", {})
    metadata = data.get("metadata", {})

    base_row = {
        "participant_folder": participant_dir.name,
        "exportId": data.get("exportId"),
        "alias": participant_info.get("alias"),
        "age": participant_info.get("age"),
        "gender": participant_info.get("gender"),
        "isNativeSpeaker": participant_info.get("isNativeSpeaker"),
        "hearingStatus": participant_info.get("hearingStatus"),
        "isListeningExpert": participant_info.get("isListeningExpert"),
        "usingHeadphones": participant_info.get("usingHeadphones"),
        "appVersion": metadata.get("appVersion"),
        "timestamp": metadata.get("timestamp"),
    }

    # user input analysis
    exp_userinput_answers = data.get("experiment_userinput", {})
    exp_userinput_scores = data.get("analysis", {}).get("experiment_userinput_scores", {})
    train_userinput_answers = data.get("training_userinput", {})
    train_userinput_scores = data.get("analysis", {}).get("training_userinput_scores", {})

    # transcript analysis
    exp_repetition_requests= data.get("analysis", {}).get("clarification_requests_experiment", {})
    exp_repetition_examples= data.get("analysis", {}).get("clarification_examples_experiment", {})
    train_repetition_requests = data.get("analysis", {}).get("clarification_requests_training", {})
    train_repetition_examples= data.get("analysis", {}).get("clarification_examples_training", {})

    # Combine all data into a single row and adds category prefixes to the questionnaire scores
    row = {
        **base_row,
        **{
            f"train_ans_{key}": value
            for key, value in train_userinput_answers.items()
        },
        **{
            f"train_score_{key}": value
            for key, value in train_userinput_scores.items()
        },
        **{
            f"exp_ans_{key}": value
            for key, value in exp_userinput_answers.items()
        },
        **{
            f"exp_score_{key}": value
            for key, value in exp_userinput_scores.items()
        },
        "exp_repetition_requests": exp_repetition_requests,
        "exp_repetition_examples": exp_repetition_examples,
        "train_repetition_requests": train_repetition_requests,
        "train_repetition_examples": train_repetition_examples,
    }

    rows.append(row)
    df = pd.DataFrame(rows)

# Compute percentage correct for training and experiment questions
train_score_cols = [c for c in df.columns if c.startswith("train_score_")]
exp_score_cols   = [c for c in df.columns if c.startswith("exp_score_")]
df["train_pct_correct"] = df[train_score_cols].sum(axis=1) / len(train_score_cols) * 100
df["exp_pct_correct"]   = df[exp_score_cols].sum(axis=1)   / len(exp_score_cols)   * 100
# Clarification requests per item (there were 4 training items and 10 experiment items)
N_TRAIN_ITEMS = 4
N_EXP_ITEMS   = 10
df["train_repetition_requests_per_item"] = df["train_repetition_requests"] / N_TRAIN_ITEMS
df["exp_repetition_requests_per_item"]   = df["exp_repetition_requests"]   / N_EXP_ITEMS

# Add a column for SNR condition based on appVersion
def extract_snr(app_version):
    if app_version is None:
        return None
    if "SNR=-5" in app_version:
        return -5
    elif "SNR=5" in app_version:
        return 5
    else:
        return None
    
df["SNR_condition"] = df["appVersion"].apply(extract_snr)

In [ ]:
df.head(10)

In [ ]:
df.columns


In [ ]:
# save to CSV
df.to_csv("all_objective_data.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Set the font size for better readability
plt.rcParams.update({'font.size': 14})

plot_data = pd.concat([
    pd.DataFrame({
        "condition": "Training\n(no noise)",
        "pct_correct": df["train_pct_correct"],
    }),
    pd.DataFrame({
        "condition": "Experiment\nSNR = +5",
        "pct_correct": df.loc[df["SNR_condition"] == 5, "exp_pct_correct"],
    }),
    pd.DataFrame({
        "condition": "Experiment\nSNR = -5",
        "pct_correct": df.loc[df["SNR_condition"] == -5, "exp_pct_correct"],
    }),
], ignore_index=True)

order = ["Training\n(no noise)", "Experiment\nSNR = +5", "Experiment\nSNR = -5"]

fig, ax = plt.subplots(figsize=(5, 5))
sns.barplot(data=plot_data, x="condition", y="pct_correct", order=order,
            palette=["lightgray", "#66c2a5", "#fc8d62"],
            errorbar="se", capsize=0.15, ax=ax)
# sns.stripplot(data=plot_data, x="condition", y="pct_correct", order=order,
              color="black", alpha=0.6, jitter=0.15, ax=ax)
ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)

ax.set_ylabel("Percent correct")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.set_title("Information Collection Score")
plt.tight_layout()
plt.show()

In [ ]:
plot_data = pd.concat([
    pd.DataFrame({
        "condition": "Training\n(no noise)",
        "clarifications": df["train_repetition_requests"],
    }),
    pd.DataFrame({
        "condition": "Experiment\nSNR = +5",
        "clarifications": df.loc[df["SNR_condition"] == 5, "exp_repetition_requests"],
    }),
    pd.DataFrame({
        "condition": "Experiment\nSNR = -5",
        "clarifications": df.loc[df["SNR_condition"] == -5, "exp_repetition_requests"],
    }),
], ignore_index=True)

order = ["Training\n(no noise)", "Experiment\nSNR = +5", "Experiment\nSNR = -5"]

fig, ax = plt.subplots(figsize=(5, 5))
sns.barplot(data=plot_data, x="condition", y="clarifications", order=order,
            palette=["lightgray", "#66c2a5", "#fc8d62"],
            errorbar="se", capsize=0.15, ax=ax)
# sns.stripplot(data=plot_data, x="condition", y="clarifications", order=order,
#               color="black", alpha=0.6, jitter=0.15, ax=ax)
ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)
ax.set_ylabel("Number of clarification requests")
ax.set_xlabel("")
ax.set_title("Number Of Clarification requests")
plt.tight_layout()
plt.show()